# 🛡️ FraudShield — Credit Card Fraud Detection
**Dataset:** [Kaggle — Credit Card Fraud Detection (mlg-ulb)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

**Model:** XGBoost + Random Forest + Logistic Regression Ensemble

**Techniques:** SMOTE, RobustScaler, Feature Engineering, SHAP

---

## 📦 Step 1: Install & Import Libraries

In [1]:
# Install required libraries
!pip install xgboost imbalanced-learn shap plotly --quiet



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\KIIT0001\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import shap
import joblib

# Style
plt.rcParams['figure.facecolor'] = '#0e1420'
plt.rcParams['axes.facecolor']   = '#141a27'
plt.rcParams['axes.edgecolor']   = '#2a3550'
plt.rcParams['text.color']       = '#e8edf7'
plt.rcParams['axes.labelcolor']  = '#e8edf7'
plt.rcParams['xtick.color']      = '#5a6580'
plt.rcParams['ytick.color']      = '#5a6580'
plt.rcParams['grid.color']       = '#1a2235'
plt.rcParams['font.family']      = 'monospace'

print('✅ All libraries imported successfully!')


✅ All libraries imported successfully!


## 📂 Step 2: Load Dataset from Kaggle

> **Download** `creditcard.csv` from 👉 https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
> Then upload it using the cell below OR mount Google Drive.

In [ ]:
# ── OPTION A: Upload manually ──────────────────────────────
from google.colab import files
print('📁 Upload your creditcard.csv file:')
uploaded = files.upload()


FileNotFoundError: [Errno 2] No such file or directory: 'creditcard.csv'

In [ ]:
# ── OPTION B: Mount Google Drive (if file is in Drive) ─────
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/creditcard.csv')

# ── Load CSV ───────────────────────────────────────────────
df = pd.read_csv('creditcard.csv')
print(f'✅ Dataset loaded!')
print(f'   Shape     : {df.shape}')
print(f'   Total Txns: {len(df):,}')
print(f'   Frauds    : {df["Class"].sum():,} ({df["Class"].mean()*100:.3f}%)')
print(f'   Legit     : {(df["Class"]==0).sum():,}')
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'creditcard.csv'

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('📊 Dataset Info:')
print(df.info())
print('\n📊 Missing values:', df.isnull().sum().sum())
print('\n📊 Class Distribution:')
print(df['Class'].value_counts())


In [ ]:
# Class imbalance pie chart
fig = go.Figure(go.Pie(
    labels=['Legitimate', 'Fraud'],
    values=df['Class'].value_counts().values,
    hole=0.55,
    marker=dict(colors=['#00e5c3', '#ff4560'],
                line=dict(color='#080b12', width=3))
))
fig.update_layout(
    title=dict(text='Class Distribution — Severe Imbalance', font=dict(color='#e8edf7', size=16)),
    paper_bgcolor='#0e1420', font=dict(color='#e8edf7')
)
fig.show()


In [ ]:
# Transaction Amount distribution by class
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Amount — Legitimate', 'Amount — Fraud'))

legit = df[df['Class']==0]['Amount']
fraud = df[df['Class']==1]['Amount']

fig.add_trace(go.Histogram(x=legit, nbinsx=80, marker_color='#00e5c3',
    name='Legit', opacity=0.8), row=1, col=1)
fig.add_trace(go.Histogram(x=fraud, nbinsx=40, marker_color='#ff4560',
    name='Fraud', opacity=0.8), row=1, col=2)

fig.update_layout(
    height=380, title='Transaction Amount Distribution by Class',
    paper_bgcolor='#0e1420', plot_bgcolor='#141a27',
    font=dict(color='#e8edf7'), showlegend=False
)
fig.show()

print(f'Legit  — Mean: ${legit.mean():.2f}  |  Max: ${legit.max():.2f}')
print(f'Fraud  — Mean: ${fraud.mean():.2f}  |  Max: ${fraud.max():.2f}')


In [ ]:
# Transactions over time (hour of day)
df['Hour'] = (df['Time'] / 3600 % 24).astype(int)

hourly = df.groupby(['Hour', 'Class']).size().reset_index(name='Count')
hourly['Type'] = hourly['Class'].map({0: 'Legit', 1: 'Fraud'})

fig = px.line(hourly, x='Hour', y='Count', color='Type',
    color_discrete_map={'Legit': '#00e5c3', 'Fraud': '#ff4560'},
    title='Transactions by Hour of Day',
    labels={'Hour': 'Hour (0-23)', 'Count': 'Transaction Count'})
fig.update_layout(paper_bgcolor='#0e1420', plot_bgcolor='#141a27',
    font=dict(color='#e8edf7'))
fig.show()


In [ ]:
# Correlation heatmap of top V features with Class
corr = df[[f'V{i}' for i in range(1,29)] + ['Amount','Class']].corr()['Class'].drop('Class').sort_values()

fig = go.Figure(go.Bar(
    x=corr.values,
    y=corr.index,
    orientation='h',
    marker=dict(
        color=corr.values,
        colorscale=[[0,'#ff4560'],[0.5,'#5a6580'],[1,'#00e5c3']]
    )
))
fig.update_layout(
    height=600, title='Feature Correlation with Fraud (Class)',
    paper_bgcolor='#0e1420', plot_bgcolor='#141a27',
    font=dict(color='#e8edf7'),
    xaxis_title='Correlation Coefficient'
)
fig.show()


## ⚙️ Step 4: Feature Engineering & Preprocessing

In [ ]:
# Feature engineering
df['Log_Amount']    = np.log1p(df['Amount'])
df['Amount_zscore'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
# Hour already added above

feature_cols = (
    [f'V{i}' for i in range(1, 29)]
    + ['Log_Amount', 'Amount_zscore', 'Hour']
)

X = df[feature_cols]
y = df['Class']

print(f'✅ Features: {len(feature_cols)}')
print(f'   {feature_cols}')


In [ ]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} samples  |  Test: {len(X_test):,} samples')
print(f'Train fraud: {y_train.sum():,}  |  Test fraud: {y_test.sum():,}')


In [ ]:
# Scale features
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('✅ Features scaled with RobustScaler')


In [ ]:
# SMOTE — oversample minority class
print('⚖️  Applying SMOTE...')
sm = SMOTE(random_state=42, k_neighbors=5)
X_res, y_res = sm.fit_resample(X_train_scaled, y_train)

print(f'Before SMOTE — Fraud: {y_train.sum():,}  Legit: {(y_train==0).sum():,}')
print(f'After  SMOTE — Fraud: {y_res.sum():,}  Legit: {(y_res==0).sum():,}')

# Visualize
fig = go.Figure(go.Bar(
    x=['Before (Legit)', 'Before (Fraud)', 'After (Legit)', 'After (Fraud)'],
    y=[(y_train==0).sum(), y_train.sum(), (y_res==0).sum(), y_res.sum()],
    marker_color=['#00e5c3','#ff4560','#00e5c3','#ff4560'],
    opacity=[0.5,0.5,1.0,1.0]
))
fig.update_layout(
    title='Class Balance: Before vs After SMOTE',
    paper_bgcolor='#0e1420', plot_bgcolor='#141a27',
    font=dict(color='#e8edf7'), height=340
)
fig.show()


## 🤖 Step 5: Train the Ensemble Model

In [ ]:
print('🤖 Building model stack...')

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=0.01,
    random_state=42
)

ensemble = VotingClassifier(
    estimators=[('xgb', xgb), ('rf', rf), ('lr', lr)],
    voting='soft',
    weights=[3, 2, 1]
)

print('🚀 Training ensemble (this may take 2–3 mins)...')
ensemble.fit(X_res, y_res)
print('✅ Training complete!')


## 📊 Step 6: Evaluate the Model

In [ ]:
y_pred  = ensemble.predict(X_test_scaled)
y_proba = ensemble.predict_proba(X_test_scaled)[:, 1]

roc    = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
prec   = precision_score(y_test, y_pred)
rec    = recall_score(y_test, y_pred)
f1     = f1_score(y_test, y_pred)

print('=' * 45)
print('  📊 MODEL EVALUATION RESULTS')
print('=' * 45)
print(f'  ROC-AUC   : {roc:.4f}')
print(f'  PR-AUC    : {pr_auc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print('=' * 45)
print()
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig = go.Figure(go.Heatmap(
    z=cm,
    x=['Predicted: Legit', 'Predicted: Fraud'],
    y=['Actual: Legit', 'Actual: Fraud'],
    colorscale=[[0,'#0e1420'],[0.5,'#1a3a6b'],[1,'#3d8bff']],
    text=cm, texttemplate='%{text}',
    textfont=dict(size=20, color='white'),
    showscale=True
))
fig.update_layout(
    title='Confusion Matrix',
    height=400,
    paper_bgcolor='#0e1420',
    font=dict(color='#e8edf7', size=13)
)
fig.show()


In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
    line=dict(color='#3d8bff', width=3),
    name=f'Ensemble (AUC = {roc:.4f})',
    fill='tozeroy', fillcolor='rgba(61,139,255,0.1)'))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
    line=dict(color='#5a6580', dash='dash'), name='Random Classifier'))
fig.update_layout(
    title=f'ROC Curve  (AUC = {roc:.4f})',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=420,
    paper_bgcolor='#0e1420', plot_bgcolor='#141a27',
    font=dict(color='#e8edf7')
)
fig.show()


In [ ]:
# Precision-Recall Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall_vals, y=precision_vals, mode='lines',
    line=dict(color='#00e5c3', width=3),
    name=f'PR Curve (AUC = {pr_auc:.4f})',
    fill='tozeroy', fillcolor='rgba(0,229,195,0.08)'
))
fig.update_layout(
    title=f'Precision-Recall Curve  (AUC = {pr_auc:.4f})',
    xaxis_title='Recall', yaxis_title='Precision',
    height=420,
    paper_bgcolor='#0e1420', plot_bgcolor='#141a27',
    font=dict(color='#e8edf7')
)
fig.show()


## 🔍 Step 7: SHAP Explainability

In [ ]:
# SHAP values for XGBoost (most interpretable)
print('🔍 Computing SHAP values (XGBoost)...')

xgb_model = ensemble.named_estimators_['xgb']
explainer  = shap.TreeExplainer(xgb_model)

# Use a sample for speed
X_sample = pd.DataFrame(X_test_scaled[:500], columns=feature_cols)
shap_values = explainer.shap_values(X_sample)

print('✅ SHAP values computed!')


In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols,
                  plot_type='bar', show=False,
                  color='#3d8bff')
plt.title('SHAP Feature Importance (XGBoost)', color='#e8edf7', fontsize=14, pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# SHAP Beeswarm plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)
plt.title('SHAP Beeswarm — Feature Impact on Fraud Prediction',
          color='#e8edf7', fontsize=13, pad=15)
plt.tight_layout()
plt.show()


## 🧪 Step 8: Predict a Single Transaction

In [ ]:
def predict_transaction(v_features: dict, amount: float, hour: int):
    """
    Predict fraud for a single transaction.

    v_features : dict with keys V1..V28
    amount     : transaction dollar amount
    hour       : hour of transaction (0–23)
    """
    row = {f'V{i}': v_features.get(f'V{i}', 0.0) for i in range(1, 29)}
    row['Log_Amount']    = np.log1p(amount)
    row['Amount_zscore'] = (amount - df['Amount'].mean()) / df['Amount'].std()
    row['Hour']          = hour

    X_input = pd.DataFrame([row])[feature_cols]
    X_scaled = scaler.transform(X_input)

    pred  = ensemble.predict(X_scaled)[0]
    proba = ensemble.predict_proba(X_scaled)[0][1]

    print('=' * 42)
    print('  🔍 FRAUD DETECTION RESULT')
    print('=' * 42)
    print(f'  Amount       : ${amount:,.2f}')
    print(f'  Hour         : {hour}:00')
    print(f'  Fraud Prob   : {proba*100:.2f}%')
    print(f'  Prediction   : {"🚨 FRAUD" if pred == 1 else "✅ LEGITIMATE"}')
    print('=' * 42)
    return pred, proba


# ── Example 1: Known fraud pattern from dataset ────────────
fraud_sample = {
    'V1': -3.043, 'V2': -3.157, 'V3': 1.088,  'V4': 2.288,
    'V5': 0.476,  'V6': -1.449, 'V7': -3.101, 'V8': -0.267,
    'V9': -1.340, 'V10': -2.792,'V11': 3.202, 'V12': -4.284,
    'V13': 0.039, 'V14': -4.432,'V15': 0.340, 'V16': -2.088,
    'V17': -5.128,'V18': -1.536,'V19': 0.294, 'V20': -0.232,
    'V21': 0.565, 'V22': -0.120,'V23': -0.380,'V24': -0.103,
    'V25': 0.468, 'V26': -0.099,'V27': 0.798, 'V28': 0.149,
}
print('--- Fraud Sample ---')
predict_transaction(fraud_sample, amount=9.99, hour=2)


In [ ]:
# ── Example 2: Normal transaction ─────────────────────────
safe_sample = {
    'V1': 1.226,  'V2': 0.141,  'V3': 0.045, 'V4': 1.203,
    'V5': 0.191,  'V6': 0.272,  'V7': 0.125, 'V8': 0.142,
    'V9': -0.059, 'V10': -0.112,'V11': -0.186,'V12': 0.133,
    'V13': -0.044,'V14': 0.094, 'V15': 0.112,'V16': -0.019,
    'V17': -0.057,'V18': 0.006, 'V19': -0.005,'V20': -0.027,
    'V21': -0.023,'V22': 0.034, 'V23': -0.010,'V24': 0.029,
    'V25': 0.031, 'V26': -0.015,'V27': 0.009, 'V28': 0.001,
}
print('--- Safe Sample ---')
predict_transaction(safe_sample, amount=149.62, hour=14)


## 💾 Step 9: Save the Model

In [ ]:
# Save model, scaler and metrics
metrics_dict = {
    'roc_auc':          round(roc, 4),
    'pr_auc':           round(pr_auc, 4),
    'precision':        round(prec, 4),
    'recall':           round(rec, 4),
    'f1':               round(f1, 4),
    'confusion_matrix': cm.tolist(),
    'feature_cols':     feature_cols,
    'total_train':      len(X_train),
    'total_test':       len(X_test),
    'fraud_train':      int(y_train.sum()),
    'fraud_test':       int(y_test.sum()),
}

joblib.dump(ensemble,     'fraud_model.pkl')
joblib.dump(scaler,       'scaler.pkl')
joblib.dump(metrics_dict, 'metrics.pkl')

print('✅ Saved: fraud_model.pkl, scaler.pkl, metrics.pkl')

# Download files to local machine
from google.colab import files
files.download('fraud_model.pkl')
files.download('scaler.pkl')
files.download('metrics.pkl')
print('📥 Files downloading to your local machine...')


---
## ✅ Summary

| Step | Description |
|------|-------------|
| 1 | Installed & imported all libraries |
| 2 | Loaded Kaggle `creditcard.csv` dataset |
| 3 | Performed EDA — class imbalance, distributions, correlations |
| 4 | Engineered features: `Log_Amount`, `Amount_zscore`, `Hour` |
| 5 | Applied SMOTE to balance fraud vs legit classes |
| 6 | Trained XGBoost + Random Forest + Logistic Regression ensemble |
| 7 | Evaluated with ROC-AUC, PR-AUC, Confusion Matrix, F1 |
| 8 | Explained predictions with SHAP feature importance |
| 9 | Saved model files for use in Streamlit app |

**Next Step:** Use `fraud_model.pkl`, `scaler.pkl`, and `metrics.pkl` with the `app.py` Streamlit GUI!